# 24장 · 문서를 바꾸며 쓰는 Streamlit RAG 챗봇

이전 앱은 늘 같은 세 파일을 검색했습니다. 새 PDF를 골랐는데 이전 판본의 쪽수가 답에 남으면 검색 품질을 평가할 수 없습니다. 학생이 실제로 문서를 바꾸고, 문단 크기를 바꾸고, 질문을 다시 보내며 데이터·검색기·화면 기록의 경계를 확인합니다.

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 처음 한 번 커널에서 `%pip install pypdf langchain-openai langchain-core chromadb python-dotenv`를 실행하고 커널을 재시작하세요. 작업 폴더의 .env에 OPENAI_API_KEY를 둡니다. 첫 색인 생성과 새 질문 임베딩은 API를 호출합니다. `%pip install streamlit langsmith`도 설치하세요.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))

## 어디까지 다시 계산해야 할까

| 바뀌는 것 | 다시 해야 하는 작업 | 재사용할 수 있는 것 |
|---|---|---|
| 질문만 바뀜 | 새 질문 임베딩, 검색, 답변 | PDF에서 읽은 쪽과 문서 색인 |
| PDF 내용 또는 분할 크기 | 페이지·청크 확인, 새 색인 준비 | 같은 설정에서 이미 만든 기존 자료 |
| 검색 방식만 변경 | 다음 질문부터 같은 후보를 재정렬할지 결정 | PDF와 문서 벡터 |

Streamlit은 화면 입력이 바뀔 때 파일을 다시 실행합니다. `st.cache_data`로 PDF 바이트·이름·분할 크기가 같을 때 계산한 쪽 목록을 재사용합니다. `st.cache_resource`에는 공용 설명서 색인 연결을 둡니다. 각 학생의 화면에 보여 줄 답변은 `session_state`에 둡니다. 이 실습의 PDF는 모두 수업 중 공유할 가상 문서만 사용합니다.

## PDF가 바뀌었는지 어떻게 알까

같은 이름의 파일을 다시 올릴 수 있으므로 파일명만 비교하면 부족합니다. 색인의 레코드는 문서 ID와 내용으로 버전을 정하고, 화면은 PDF 바이트의 지문과 설정을 봅니다. 지문은 파일 내용에서 계산한 식별값입니다. 문단 크기도 화면 기준에 포함해 크기를 바꾸면 이전 답변을 비웁니다.

In [2]:
import hashlib
from rag_lab import MANUALS
for file in MANUALS:
    print(file.name, hashlib.sha256(file.read_bytes()).hexdigest()[:12])

recorder-a-1.0.pdf 50841897b4ba
recorder-a-1.1.pdf 45d300f6709b
recorder-b-1.0.pdf c7b60ef10fa2


## 앱 파일 만들기

아래 셀을 한 번 실행하고 같은 폴더의 터미널에서 `python -m streamlit run manual_chat.py`를 실행하세요. 기본으로 세 파일이 선택됩니다. 공용 파일 한 개만 골라 판본 차이를 검증하거나, 실습으로 받은 새 PDF를 올릴 수 있습니다. 업로드하면 그 문서 한 개로 질문합니다. 같은 옵션에서 질문을 두 번 보내도 이전 대화가 이번 질문에 자동으로 포함되지는 않습니다. 기본 모드는 뒤의 평가 수업과 같은 기준을 유지해 후보 네 쪽을 모두 전달합니다. 재정렬 모드는 23장에서 검토한 상위 두 쪽을 전달합니다. 순서의 효과만 비교할 때에는 두 방식 모두 두 쪽을 보내는 23장 앱을 사용하세요.

In [3]:
# 이 셀의 문자열이 앱 파일의 실제 내용입니다.
app_code = r'''import hashlib
import io
import sys
from pathlib import Path
import streamlit as st
from pypdf import PdfReader
from langsmith import traceable, get_current_run_tree

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pre-mid/documents/rag").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/rag"))
from rag_lab import MANUALS, page_chunks, open_index, retrieve, rerank, answer_from_documents

@st.cache_data
def read_pdf(name, data, size):
    # 바이트가 바뀌면 이름이 같아도 새로 읽습니다.
    reader = PdfReader(io.BytesIO(data))
    rows = []
    for number, page in enumerate(reader.pages, 1):
        text = (page.extract_text() or "").strip()
        if not text:
            raise ValueError(f"{name} {number}쪽에 추출할 텍스트가 없습니다.")
        official = next((f for f in MANUALS if f.name == name and f.read_bytes() == data), None)
        if official:
            _, model, edition = official.stem.split("-")
            model = model.upper()
        else:
            model, edition = "수업 자료", name
        rows.append(dict(id=f"{name}#p{number}", source=name, page=number,
                         model=model, edition=edition, text=text))
    return page_chunks(rows, size or None)

@st.cache_resource
def shared_index(rows):
    return open_index(rows)

@traceable(name="설명서 질문 한 건", process_inputs=lambda inputs: {k:v for k,v in inputs.items() if k != "index"})
def ask(index, question, mode):
    docs = retrieve(index, question, k=4)
    ranked = rerank(question, docs)[:2] if mode == "재정렬" else docs
    answer = answer_from_documents(question, ranked)
    run = get_current_run_tree()
    return {"question":question, "answer":answer, "initial":[d["id"] for d in docs],
            "documents":ranked, "trace":str(run.id) if run else ""}

st.title("설명서와 근거를 함께 읽는 챗봇")
files = st.multiselect("공용 설명서", MANUALS, default=list(MANUALS), format_func=lambda p:p.name)
upload = st.file_uploader("또는 수업용 PDF 한 개 올리기", type=["pdf"])
size = st.selectbox("한 조각 길이 (0은 한 쪽)", [0, 100, 180])
mode = st.radio("후보 정렬", ["기본", "재정렬"], horizontal=True)
inputs = [(upload.name, upload.getvalue())] if upload else [(f.name, f.read_bytes()) for f in files]
signature = (tuple((name,hashlib.sha256(data).hexdigest()) for name,data in inputs), size)
if st.session_state.get("signature") != signature:
    st.session_state.signature = signature
    st.session_state.runs = []
question = st.chat_input("자료의 기종·판본을 넣어 질문하세요.", disabled=not inputs)
if question:
    rows = [r for name,data in inputs for r in read_pdf(name, data, size)]
    index = shared_index(rows)
    result = ask(index, question, mode)
    result["index"] = index[0].name
    st.session_state.runs.append(result)
for result in st.session_state.runs:
    st.chat_message("user").write(result["question"])
    st.chat_message("assistant").write(result["answer"])
    with st.expander("검색한 문단과 출처"):
        st.write("첫 후보",result["initial"],"색인",result["index"])
        for row in result["documents"]:
            st.write(row["id"], row["text"])
        if result["trace"]:
            st.write("실행 ID",result["trace"])
'''
Path("manual_chat.py").write_text(app_code, encoding="utf-8")
print("manual_chat.py 생성")

manual_chat.py 생성


## 새 문서로 바뀌었는지 검수하기

1. A형 1.0판만 선택하고 “계속 녹음하면 파일은 몇 분마다 나뉘나요?”를 보냅니다. 답과 파일명을 기록합니다.
2. 같은 화면에서 A형 1.1판만 남깁니다. 이전 대화가 비워진 것을 본 뒤 같은 질문을 보냅니다. 30분과 45분이 달라진 이유를 인용 쪽에서 찾습니다.
3. 입력 그대로 분할 길이만 100으로 바꾸세요. 새 ID 끝에 c1처럼 조각 번호가 붙는지 확인합니다. 정답 문구가 끊겼다면 길이를 조정합니다.
4. `loan-summer.pdf`를 올리고 승인된 신청의 시간 변경을 묻습니다. 펼친 문단에는 장비 대여 안내의 파일명만 있어야 합니다. 기종 설명서가 남으면 화면 상태와 실제 전달한 자료를 따로 확인합니다.

## LangSmith에서 실제 실행 한 건 열기

LangSmith는 앱 실행 중 각 모델 호출의 입력과 출력을 살펴보는 도구입니다. 수업용 프로젝트에서 키를 준비하고 작업 폴더의 .env에 `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT=recorder-class`를 지정한 뒤 앱 프로세스를 다시 시작합니다. 수업용 질문을 하나 새로 보내고 해당 프로젝트에서 **설명서 질문 한 건**을 엽니다. 앱에 보인 실행 ID, 질문과 상위 문단 목록, 실제 answer를 나란히 확인합니다. 실행 기록에는 질문·문서 내용이 전송되므로 가상 수업 자료로 진행하세요. 계정 없이도 로컬 검색과 화면 검증은 할 수 있습니다.

## 파일 이름이 같아도 내용을 바꾸어 검수하기

학급 자료 폴더에서 3월 대여 안내와 4월 대여 안내의 복사본을 각각 다른 폴더에 저장하고 이름을 둘 다 guide.pdf로 바꿔 보세요. 앱에서는 순서대로 한 개씩 올립니다. 같은 이름이라도 답이 3일에서 5일로 바뀌려면 새 내용으로 쪽과 색인을 준비해야 합니다. 먼저 코드로 두 파일의 지문이 다른지 확인합니다.

In [4]:
for original_name in ["loan-spring.pdf", "loan-summer.pdf"]:
    data = (rag_dir / original_name).read_bytes()
    print("올려 보일 이름: guide.pdf", "원본", original_name,
          "지문", hashlib.sha256(data).hexdigest()[:12])

올려 보일 이름: guide.pdf 원본 loan-spring.pdf 지문 d23b12989b3b
올려 보일 이름: guide.pdf 원본 loan-summer.pdf 지문 a52110fe2f55


이름만 캐시 키로 쓰면 첫 문서의 값을 재사용할 수 있습니다. 앱의 read_pdf는 이름과 실제 바이트와 조각 크기를 모두 인자로 받으므로 둘을 구별합니다. 지문이 달라져도 화면에 남아 있던 이전 답을 지우지 않으면 학생이 다른 판본의 답을 읽을 수 있습니다. 두 가지를 따로 시험합니다.

화면이 갱신됐다는 사실과 새 질문이 전송됐다는 사실을 구별하세요. 모드 위젯만 바꾼 뒤 모델 사용량이 늘거나 이전 질문에 추가 답변이 생기면, 모델 호출이 입력 조건 바깥에 있는지 봅니다. [Streamlit 캐시 참고](https://docs.streamlit.io/develop/concepts/architecture/caching) · [LangSmith 추적 참고](https://docs.langchain.com/langsmith/trace-with-langchain)

## 실습 · 다른 학생에게 기준선 넘기기

짝에게 PDF·질문·설정만 건네 새 세션에서 다시 실행하게 하세요. 직접 답을 찾을 수 있는 D04와 답을 보류할 D05를 모두 사용합니다. 문서명, 분할 길이, 검색 방식, 색인 이름, 실제 후보 순위, 실제 인용과 답변, LangSmith 사용 시 실행 ID를 기록합니다. 둘의 답이 다르면 후보가 처음 달라진 지점을 찾습니다.

In [5]:
# 아래 셀에서 D04의 원본 판본·쪽을 확인하고 실제 화면의 인용과 맞춰 보세요.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>from rag_lab import SOURCES, read_pages
page = read_pages(SOURCES / &quot;recorder-a-1.1.pdf&quot;)[1]
print(page[&quot;id&quot;], page[&quot;text&quot;][48:139])</code></pre><p>D04는 A형 1.1판 2쪽의 45분과 녹음 지속이 함께 나와야 합니다. D05는 문서에 무선 전송 지원을 확정할 근거가 없습니다. 표에는 각자 실제로 받은 결과를 기록합니다. 실행 ID는 실제로 추적을 켠 실행에서 복사하세요.</p><p>집필 시 실행 결과</p><pre>recorder-a-1.1.pdf#p2 027년 4월 1일부터
연속 녹음
한 번에 길게 녹음하면 45분마다 새 파일이 열립니다. 파일이
바뀌어도 녹음은 계속됩니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로</pre></details>

## 독립 실습 · 새 PDF를 받는 교대 담당자

짝이 어떤 파일을 선택했는지 가린 채 질문을 하나 보냅니다. 답과 근거만으로 **어떤 문서 버전**을 썼는지 확인할 수 있는지 묻습니다. 이어 3월 대여 안내와 4월 대여 안내를 번갈아 업로드하고 같은 신규 신청 문의를 보냅니다. 마지막에는 이미 승인된 신청의 시간 변경에 대한 질문을 새로 입력하세요. 녹음기 색인이나 이전 업로드 정보가 다시 나타나는지 봅니다.

| 파일·설정 | 색인 이름 | 실제 질문과 답 | 실제 인용 | 확인한 원문 | 실행 ID(추적한 경우) |
|---|---|---|---|---|---|
| 교대 전 | | | | | |
| 교대 후 | | | | | |

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>3월 안내 업로드 시 3일, 4월 안내에서는 5일이 나와야 하며 각각 업로드한 파일의 1쪽을 인용해야 합니다. 이미 승인된 수령 시간만 바꾸는 질문은 2쪽의 전날 오후 4시입니다. 새 답이 맞아도 이전 파일의 근거가 화면에 남으면 교대 담당자가 어떤 자료로 판단했는지 혼동합니다. 파일 변경 직후 기록이 비고, 새로 물은 뒤 표시된 실제 후보·색인 이름이 현재 파일과 연결되는지 확인합니다.</p></details>

## 다음 수업에 가져갈 것

세 설명서·색인 설정·D01~D05의 검색 결과·실행 한 건을 팀에 넘깁니다. 25장에서는 2부의 모델 호출 함수를 다시 꺼내 문의의 조건을 추출하고 초안을 씁니다. 5부에서 지금의 검색기와 답변 함수를 그래프 노드로 연결하고 6부에서 이 기준 결과와 비교합니다.